In [1]:
import os

from dotenv import load_dotenv
load_dotenv()
print(os.getenv('GROQ_API_KEY'))

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


In [ ]:
import re
import os
import time
import random
import base64

import requests
from urllib.parse import urlparse
import concurrent.futures

from dotenv import load_dotenv
from huggingface_hub import InferenceClient

import mysql.connector
from mysql.connector import Error

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By

In [2]:
brave_path = r'C:\Program Files\BraveSoftware\Brave-Browser-Nightly\Application\brave.exe'
base_dir = r'e:\CCNY\DSE I2400 - Data Engineering\Project 1\Custom Bot'

In [3]:
search_engines = {
    'Google': 'https://www.google.com/search?q=',
    'Bing': 'https://www.bing.com/search?q=',
    'Yahoo': 'https://search.yahoo.com/search?p=',
    'DuckDuckGo': 'https://duckduckgo.com/?q='
}

In [49]:
# # def run_headless_ingestion(search_term, engine_name):
#     if engine_name not in search_engines:
#         print(f'Error: Search engine \'{engine_name}\' is not supported.')
#         return None

#     options = uc.ChromeOptions()
#     options.binary_location = brave_path
    
#     options.add_argument('--headless=new')
#     options.add_argument('--disable-blink-features=AutomationControlled')
#     options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36') 
    
#     driver = uc.Chrome(options=options, browser_executable_path=brave_path)
    
#     try:
#         safe_term = search_term.replace(' ', '+')
#         url = f'{search_engines[engine_name]}{safe_term}'
        
#         driver.get(url)
#         time.sleep(random.uniform(4, 6)) 

#         web_page_height = driver.execute_script('''
#             return Math.max(
#                 document.body.scrollHeight, 
#                 document.body.offsetHeight, 
#                 document.documentElement.clientHeight, 
#                 document.documentElement.scrollHeight, 
#                 document.documentElement.offsetHeight
#             );
#         ''')

#         driver.set_window_size(1920, web_page_height)
#         time.sleep(2) 

#         filename = f'screenshot_{engine_name.lower()}.png'
#         filepath = os.path.join(base_dir,'screenshots', filename)
        
#         driver.save_screenshot(filepath)
        
#         # print(f'Captured {filename} - Total Height: {web_page_height}px')
#         print(f'Capturing screenshot for {engine_name}')

#     finally:
#         driver.quit()
        
#     return filename

In [50]:
def run_headless_ingestion(search_term, engine_name, page_no=1):
    if engine_name not in search_engines:
        print(f'Error: Search engine \'{engine_name}\' is not supported.')
        return None

    options = uc.ChromeOptions()
    options.binary_location = brave_path
    
    options.add_argument('--headless=new')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1920,1080')
    
    profile_path = os.path.join(os.getcwd(), 'chrome_profile')
    options.add_argument(f'--user-data-dir={profile_path}')
    
    user_agents = [
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    ]
    options.add_argument(f'--user-agent={random.choice(user_agents)}')
    
    driver = uc.Chrome(options=options, browser_executable_path=brave_path, version_main=146)
    
    try:
        safe_term = search_term.replace(' ', '+')
        url = f'{search_engines[engine_name]}{safe_term}'
        
        if page_no > 1:
            if engine_name == 'Google':
                url += f'&start={(page_no - 1) * 10}'
            elif engine_name == 'Bing':
                url += f'&first={(page_no - 1) * 5 + 1}&FORM=PERE'
            elif engine_name == 'Yahoo':
                url += f'&b={(page_no - 1) * 10 + 1}'
        
        driver.get(url)
        time.sleep(random.uniform(5, 8)) 

        if engine_name == 'DuckDuckGo' and page_no > 1:
            for _ in range(page_no - 1):
                driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
                time.sleep(3)
                try:
                    driver.execute_script('var btn = document.getElementById(\'more-results\') || document.querySelector(\'.result--more__btn\'); if(btn) btn.click();')
                    time.sleep(3)
                except Exception:
                    pass

        web_page_height = driver.execute_script('''
            return Math.max(
                document.body.scrollHeight, 
                document.body.offsetHeight, 
                document.documentElement.clientHeight, 
                document.documentElement.scrollHeight, 
                document.documentElement.offsetHeight
            );
        ''')

        driver.set_window_size(1920, web_page_height)
        time.sleep(random.uniform(2, 4)) 

        filename = f'screenshot_{engine_name.lower()}_{page_no}.png'
        filepath = os.path.join(base_dir, 'screenshots', filename)
        
        driver.save_screenshot(filepath)
        
        print(f'Capturing screenshot for {engine_name} Page {page_no}')

    finally:
        driver.quit()
        
    return filepath

In [9]:
load_dotenv()
hf_token = os.getenv('HF_TOKEN')

client = InferenceClient(
    model='Qwen/Qwen2.5-VL-7B-Instruct',
    token=hf_token
)

In [6]:
def extract_links(filepath):
    with open(filepath, 'rb') as f:
        img_b64 = base64.b64encode(f.read()).decode('utf-8')
    
    prompt = (
        'Identify the search results in this image. Extract the main website domain or URL for each result. '
        'Format each strictly as a clean URL starting with https://. Do not include trailing punctuation, '
        'colons, or breadcrumb arrows. If the image shows a CAPTCHA page, ignore it and return nothing. '
        'If there is a cookie request modal or popup, ignore it and only extract the underlying search result URLs.'
    ) 
    
    messages = [
        {
            'role': 'user',
            'content': [
                {
                    'type': 'image_url',
                    'image_url': {'url': f'data:image/png;base64,{img_b64}'}
                },
                {
                    'type': 'text',
                    'text': prompt
                }
            ]
        }
    ]
    
    try:
        response = client.chat_completion(
            messages=messages,
            max_tokens=500
        )
        
        raw_text = response.choices[0].message.content
        
        urls = re.findall(r'https?://[a-zA-Z0-9.\-/?=&#_]+', raw_text)
        
        unique_urls = list(set([url.rstrip('.-') for url in urls]))
        
        print(f'Extracted {len(unique_urls)} URLs from {filepath.split(os.sep)[-1]}')
        return unique_urls
            
    except Exception as e:
        print(f'Extraction failed for {filepath}: {e}')
        return []

In [53]:
queries = [
    'Childhood cancer morbidity rate',
    'Childhood cancer early diagnosis methods',
    'Childhood cancer immunotherapy success rate',
    'Childhood cancer treatment best hospitals USA',
    'Childhood cancer treatment best hospitals Europe Asia Latin countries'
]

In [54]:
search_term = 'Childhood cancer treatment best hospitals US'

engines = list(search_engines.keys())
screenshot_files = []

for e in engines:
    for page in range(1, 3):
        filepath = run_headless_ingestion(search_term, e, page_no=page)
        screenshot_files.append(filepath)
        
screenshot_files

Capturing screenshot for Google Page 1
Capturing screenshot for Google Page 2
Capturing screenshot for Bing Page 1
Capturing screenshot for Bing Page 2
Capturing screenshot for Yahoo Page 1
Capturing screenshot for Yahoo Page 2
Capturing screenshot for DuckDuckGo Page 1
Capturing screenshot for DuckDuckGo Page 2


['e:\\CCNY\\DSE I2400 - Data Engineering\\Project 1\\Custom Bot\\screenshots\\screenshot_google_1.png',
 'e:\\CCNY\\DSE I2400 - Data Engineering\\Project 1\\Custom Bot\\screenshots\\screenshot_google_2.png',
 'e:\\CCNY\\DSE I2400 - Data Engineering\\Project 1\\Custom Bot\\screenshots\\screenshot_bing_1.png',
 'e:\\CCNY\\DSE I2400 - Data Engineering\\Project 1\\Custom Bot\\screenshots\\screenshot_bing_2.png',
 'e:\\CCNY\\DSE I2400 - Data Engineering\\Project 1\\Custom Bot\\screenshots\\screenshot_yahoo_1.png',
 'e:\\CCNY\\DSE I2400 - Data Engineering\\Project 1\\Custom Bot\\screenshots\\screenshot_yahoo_2.png',
 'e:\\CCNY\\DSE I2400 - Data Engineering\\Project 1\\Custom Bot\\screenshots\\screenshot_duckduckgo_1.png',
 'e:\\CCNY\\DSE I2400 - Data Engineering\\Project 1\\Custom Bot\\screenshots\\screenshot_duckduckgo_2.png']

In [10]:
screenshot_dir = 'screenshots'
screenshot_files = [
    os.path.join(screenshot_dir, f) 
    for f in os.listdir(screenshot_dir) 
    if os.path.isfile(os.path.join(screenshot_dir, f)) and f.endswith('.png')
]

screenshot_files

['screenshots\\screenshot_bing_1.png',
 'screenshots\\screenshot_bing_2.png',
 'screenshots\\screenshot_duckduckgo_1.png',
 'screenshots\\screenshot_duckduckgo_2.png',
 'screenshots\\screenshot_google_1.png',
 'screenshots\\screenshot_google_2.png',
 'screenshots\\screenshot_yahoo_1.png',
 'screenshots\\screenshot_yahoo_2.png']

In [11]:
urls = []

for screenshot in screenshot_files:
    urls.extend(extract_links(screenshot))

Extracted 10 URLs from screenshot_bing_1.png
Extracted 6 URLs from screenshot_bing_2.png
Extracted 13 URLs from screenshot_duckduckgo_1.png
Extracted 23 URLs from screenshot_duckduckgo_2.png
Extracted 8 URLs from screenshot_google_1.png
Extracted 9 URLs from screenshot_google_2.png
Extracted 12 URLs from screenshot_yahoo_1.png
Extracted 8 URLs from screenshot_yahoo_2.png


In [12]:
for url in urls:
    print(url)

https://www.stjude.org
https://www.acibademhealthpoint.com
https://www.oncologynewscentral.com
https://www.chla.org
https://www.uclahealth.org
https://www.pmnewswire.com
https://www.beckershospitalreview.com
https://www.acco.org
https://health.usnews.com
https://rankings.newsweek.com
https://my.clevelandclinic.org
https://www.medeboundhealth.com
https://bloodcancerunited.org
https://www.uvmhealth.org
https://kidshealth.org
https://www.childrenscolorado.org
https://www.stjude.org
https://www.acibademhealthpoint.com
https://www.oncologynewscentral.com
https://www.chla.org
https://rankings.statista.com
https://www.huntingtonhealth.org
https://www.prnewswire.com
https://us-uk.bookimed.com
https://www.envita.com
https://www.beckershospitalreview.com
https://www.nationwidechildrens.org
https://health.usnews.com
https://www.rankings.newsweek.com
https://kidshealth.org
https://rankings.newweek.com
https://www.cedars-sinai.org
https://www.nationwidechildrens.org
https://www.clevelandclinic.org


In [13]:
print(f'Total URLs extracted: {len(urls)}')
print(f'Unique URLs extracted: {len(set(urls))}')

Total URLs extracted: 89
Unique URLs extracted: 59


In [ ]:
unique_urls = set(urls)
# unique_urls

{'https://bloodcancerunited.org',
 'https://health.usnews.com',
 'https://kidshealth.org',
 'https://kidshealth.org/en/',
 'https://medeboundhealth.com/post/',
 'https://my.clevelandclinic.org',
 'https://my.clevelandclinic.org/',
 'https://nih.gov/research/centers-programs/nih-robert-rosenberg',
 'https://pennstatehealthnews.org',
 'https://rankings.newsweek.com',
 'https://rankings.newweek.com',
 'https://rankings.statista.com',
 'https://us-bookimed.com',
 'https://www.acco.org',
 'https://www.acibademhealthpoint.com',
 'https://www.beckershospitalreview.com',
 'https://www.bloodcancerunited.org/',
 'https://www.cancer.gov',
 'https://www.cancer.org',
 'https://www.cancerrounds.com',
 'https://www.cancerrounds.com/hospitals/best-childrens-hospitals-in-dubai-uae-2026',
 'https://www.cedars-sinai.org',
 'https://www.cedars-sinai.org/',
 'https://www.childrenscolorado.org',
 'https://www.childrenscolorado.org/',
 'https://www.childrenslhospital.org',
 'https://www.childrensnational.org

In [41]:
def process_single_url(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    try:
        if not url.startswith('http'):
            url = 'https://' + url

        parsed = urlparse(url)
        netloc = parsed.netloc.lower()

        if netloc.startswith('www.'):
            netloc = netloc[4:]

        path = parsed.path
        if path.endswith('/'):
            path = path[:-1]

        clean_url = f'https://{netloc}{path}'
        if parsed.query:
            clean_url += f'?{parsed.query}'

        response = requests.head(clean_url, headers=headers, timeout=5, allow_redirects=True)
        
        if response.status_code < 404 or response.status_code == 405:
            print(f'Validated URL: {clean_url} (Status: {response.status_code})')
            return clean_url
            
    except Exception:
        pass
        
    return None

In [40]:
def clean_and_validate_urls(raw_urls):
    raw_urls = set(raw_urls)
    valid_urls = []
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
        results = executor.map(process_single_url, raw_urls)
        
        for result in results:
            if result:
                valid_urls.append(result)
                
    return valid_urls

In [42]:
print('raw urls:', len(urls))
clean_urls = clean_and_validate_urls(urls)
print('cleaned urls:', len(clean_urls))

raw urls: 89
Validated URL: https://medicalxpress.com (Status: 200)
Validated URL: https://rankings.newsweek.com (Status: 200)
Validated URL: https://rankings.newsweek.com (Status: 200)
Validated URL: https://my.clevelandclinic.org (Status: 200)
Validated URL: https://kidshealth.org (Status: 200)
Validated URL: https://uclahealth.org (Status: 200)
Validated URL: https://chla.org (Status: 403)
Validated URL: https://pennstatehealthnews.org (Status: 200)
Validated URL: https://beckershospitalreview.com (Status: 403)
Validated URL: https://oncologynewscentral.com (Status: 403)
Validated URL: https://msccc.org (Status: 405)
Validated URL: https://us-uk.bookimed.com (Status: 200)
Validated URL: https://youtube.com (Status: 200)
Validated URL: https://psu.edu (Status: 200)
Validated URL: https://stjude.org (Status: 200)
Validated URL: https://medeboundhealth.com/post/best-hospitals-for-breast-cancer-in-the-us (Status: 200)
Validated URL: https://nationwidechildrens.org (Status: 200)
Validate

In [32]:
clean_urls

['https://chla.org',
 'https://youtube.com',
 'https://beckershospitalreview.com',
 'https://medicalxpress.com',
 'https://eurekalert.org',
 'https://psu.edu',
 'https://uclahealth.org',
 'https://kidshealth.org',
 'https://stjude.org',
 'https://dana-farber.org',
 'https://cedars-sinai.org',
 'https://nationwidechildrens.org',
 'https://pennstatehealthnews.org',
 'https://clevelandclinic.org',
 'https://rankings.newsweek.com',
 'https://rankings.newsweek.com',
 'https://my.clevelandclinic.org',
 'https://oncologynewscentral.com',
 'https://medeboundhealth.com/post/best-hospitals-for-breast-cancer-in-the-us',
 'https://rankings.statista.com',
 'https://msccc.org',
 'https://us-uk.bookimed.com',
 'https://envita.com',
 'https://chop.edu',
 'https://stjude.org',
 'https://acibademhealthpoint.com',
 'https://cancer.gov',
 'https://medeboundhealth.com',
 'https://digitaljournal.com',
 'https://chla.org',
 'https://huntingtonhealth.org',
 'https://uvmhealth.org',
 'https://uclahealth.org',


In [58]:
def get_db_connection():
    try:
        connection = mysql.connector.connect(
            host='localhost',
            user='root',
            password='00000',
            database='my_custom_bot'
        )
        return connection
    except mysql.connector.Error as e:
        print(f'Database connection failed: {e}')
        return None

In [63]:
conn = get_db_connection()

In [60]:
def insert_clean_urls(connection, search_term_id, clean_urls):
    cursor = connection.cursor()
    query = 'INSERT INTO clean_urls (search_term_id, url) VALUES (%s, %s)'
    data = [(search_term_id, url) for url in clean_urls]
    
    try:
        cursor.executemany(query, data)
        connection.commit()
        print(f'Successfully inserted {cursor.rowcount} urls')
    except Exception as e:
        connection.rollback()
        print(f'Database insertion failed: {e}')
    finally:
        cursor.close()

In [64]:
insert_clean_urls(conn, 1, clean_urls)

Database insertion failed: 1452 (23000): Cannot add or update a child row: a foreign key constraint fails (`my_custom_bot`.`clean_urls`, CONSTRAINT `clean_urls_ibfk_1` FOREIGN KEY (`search_term_id`) REFERENCES `search_terms` (`id`))
